# Watermark Removal with Deep Learning
## U-Net Architecture for Image Inpainting

**Dataset Structure:**
```
wm-nown/
├── train/
│   ├── watermark/      (641 watermarked images)
│   └── no-watermark/   (641 clean images)
├── test/
│   ├── watermark/      (81 watermarked images)
│   └── no-watermark/   (81 clean images)
├── train_pairs.csv     (Maps watermarked → clean)
└── test_pairs.csv      (Maps watermarked → clean)
```

**Model Architecture:**
- **U-Net** with encoder-decoder structure
- **Skip connections** to preserve spatial information
- **Multi-loss function**: L1 + Perceptual (VGG19) + SSIM
- **Data augmentation** for robust training

**Expected Performance:**
- PSNR: 30-35 dB
- SSIM: 0.90-0.95
- Training time: 2-4 hours on GPU

---
## 1. Setup and Configuration

In [ ]:
# ============================================================================
# IMPORTS
# ============================================================================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

# TensorFlow and Keras
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

print(f"TensorFlow version: {tf.__version__}")

In [ ]:
# ============================================================================
# LOAD VGG19 FOR PERCEPTUAL LOSS (GPU + INTERNET)
# ============================================================================
# VGG19 is used for perceptual loss to improve visual quality
# Requires ~550 MB download from internet

USE_PERCEPTUAL_LOSS = False
vgg_model = None

print("Attempting to load VGG19 for perceptual loss...")
print("This requires internet access and will download ~550 MB")
print()

try:
    from tensorflow.keras.applications import VGG19
    
    # Download and load VGG19 with ImageNet weights
    print("Downloading VGG19 weights from ImageNet...")
    vgg_model = VGG19(
        include_top=False,
        weights='imagenet',
        input_shape=(256, 256, 3)
    )
    vgg_model.trainable = False
    
    USE_PERCEPTUAL_LOSS = True
    
    print("
" + "="*70)
    print("✓ VGG19 LOADED SUCCESSFULLY!")
    print("="*70)
    print("Perceptual loss: ENABLED")
    print("Loss function: L1 + SSIM + Perceptual")
    print("Expected PSNR: 30-35 dB (Outstanding quality!)")
    print("Expected SSIM: 0.90-0.95")
    print("="*70)
    
except Exception as e:
    print("
" + "="*70)
    print("⚠ VGG19 LOADING FAILED")
    print("="*70)
    print(f"Error: {type(e).__name__}: {e}")
    print()
    print("This usually means:")
    print("  1. Internet access is disabled in notebook settings")
    print("  2. Network connection issues")
    print("  3. Kaggle's download server is temporarily unavailable")
    print()
    print("SOLUTION: Enable internet access")
    print("  1. Click ⚙️ (Settings) in right sidebar")
    print("  2. Toggle 'Internet' to ON")
    print("  3. Click 'Restart & Run All'")
    print()
    print("Fallback mode: Training will continue without perceptual loss")
    print("Loss function: L1 + SSIM only")
    print("Expected PSNR: 28-33 dB (Still excellent quality!)")
    print("Expected SSIM: 0.88-0.93")
    print("="*70)
    print()
    print("Note: Results are still excellent without VGG19!")
    print("The difference is small (~2-3 dB PSNR).")
    print("="*70)

In [ ]:
# ============================================================================
# GPU CONFIGURATION
# ============================================================================
# Enable GPU memory growth to prevent OOM errors
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        # Enable memory growth for all GPUs
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"✓ GPU available: {len(gpus)} device(s)")
        print(f"  Devices: {[gpu.name for gpu in gpus]}")
    except RuntimeError as e:
        print(f"GPU setup error: {e}")
else:
    print("⚠ No GPU found, training will use CPU (much slower)")
    print("  Go to Settings → Accelerator → GPU T4 x2")

In [ ]:
# ============================================================================
# CONFIGURATION
# ============================================================================
CONFIG = {
    # Image settings
    'IMAGE_SIZE': (256, 256),    # Input/output image dimensions
    
    # Training hyperparameters
    'BATCH_SIZE': 16,            # Images per batch (reduce if OOM)
    'EPOCHS': 100,               # Number of training epochs
    'LEARNING_RATE': 1e-4,       # Adam optimizer learning rate
    
    # Reproducibility
    'SEED': 42                   # Random seed for consistency
}

# Set random seeds for reproducible results
np.random.seed(CONFIG['SEED'])
tf.random.set_seed(CONFIG['SEED'])

print("Configuration:")
for key, value in CONFIG.items():
    print(f"  {key:20s}: {value}")

---
## 2. Dataset Loading and Exploration

In [ ]:
# ============================================================================
# DATASET PATHS
# ============================================================================
# IMPORTANT: Update BASE_PATH to match your Kaggle dataset location
# If you uploaded the dataset as "my-watermark-dataset", change to:
# BASE_PATH = '/kaggle/input/my-watermark-dataset'

BASE_PATH = '/kaggle/input/watermark-removal-dataset'  # ← UPDATE THIS!

# Dataset folder (contains train/, test/, and CSV files)
DATASET_DIR = os.path.join(BASE_PATH, 'wm-nown')

# CSV files with image pair mappings
TRAIN_CSV = os.path.join(DATASET_DIR, 'train_pairs.csv')
TEST_CSV = os.path.join(DATASET_DIR, 'test_pairs.csv')

# Verify paths exist
print("Checking dataset paths...")
print(f"  Dataset directory: {DATASET_DIR}")
print(f"  Exists: {os.path.exists(DATASET_DIR)}")

if not os.path.exists(DATASET_DIR):
    print("\n⚠ ERROR: Dataset directory not found!")
    print("Available paths:")
    os.system('ls -la /kaggle/input/')
    raise FileNotFoundError(f"Dataset not found at {DATASET_DIR}")

In [ ]:
# ============================================================================
# LOAD CSV FILES
# ============================================================================
# CSV format:
#   - original_filename: Original image name
#   - new_filename: Renamed image name
#   - watermark_path: Path to watermarked image (e.g., 'train/watermark/image_0001.jpg')
#   - clean_path: Path to clean image (e.g., 'train/no-watermark/image_0001.jpg')

print("Loading CSV files...")
train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)

print(f"\n✓ Training pairs: {len(train_df)}")
print(f"✓ Testing pairs: {len(test_df)}")
print(f"\nCSV columns: {train_df.columns.tolist()}")
print(f"\nFirst 3 training examples:")
print(train_df[['original_filename', 'watermark_path', 'clean_path']].head(3))

In [ ]:
# ============================================================================
# VISUALIZE SAMPLE IMAGE PAIRS
# ============================================================================
def show_image_pairs(df, base_dir, num_pairs=3):
    """
    Display watermarked and clean image pairs side by side.
    
    Args:
        df: DataFrame with 'watermark_path' and 'clean_path' columns
        base_dir: Base directory containing the image folders
        num_pairs: Number of pairs to display
    """
    fig, axes = plt.subplots(num_pairs, 2, figsize=(12, 4*num_pairs))
    
    for idx in range(num_pairs):
        row = df.iloc[idx]
        
        # Load images using relative paths from CSV
        wm_path = os.path.join(base_dir, row['watermark_path'])
        clean_path = os.path.join(base_dir, row['clean_path'])
        
        wm_img = Image.open(wm_path)
        clean_img = Image.open(clean_path)
        
        # Display watermarked image
        axes[idx, 0].imshow(wm_img)
        axes[idx, 0].set_title(f'Watermarked (Input)\n{row["original_filename"]}', fontsize=11)
        axes[idx, 0].axis('off')
        
        # Display clean image
        axes[idx, 1].imshow(clean_img)
        axes[idx, 1].set_title('Clean (Target)', fontsize=11)
        axes[idx, 1].axis('off')
    
    plt.tight_layout()
    plt.show()

# Show sample pairs from training set
print("Sample image pairs from training set:")
show_image_pairs(train_df, DATASET_DIR, num_pairs=3)

---
## 3. Data Pipeline with Augmentation

In [ ]:
# ============================================================================
# IMAGE LOADING AND PREPROCESSING
# ============================================================================
def load_image_pair(watermarked_path, clean_path, img_size=(256, 256)):
    """
    Load and preprocess a watermarked-clean image pair.
    
    Steps:
    1. Load JPEG images
    2. Decode to RGB (3 channels)
    3. Resize to target size
    4. Normalize to [-1, 1] range
    
    Args:
        watermarked_path: Path to watermarked image
        clean_path: Path to clean image
        img_size: Target image size (height, width)
    
    Returns:
        (wm_img, clean_img): Tuple of preprocessed tensors
    """
    # Load watermarked image
    wm_img = tf.io.read_file(watermarked_path)
    wm_img = tf.image.decode_jpeg(wm_img, channels=3)
    
    # Load clean image
    clean_img = tf.io.read_file(clean_path)
    clean_img = tf.image.decode_jpeg(clean_img, channels=3)
    
    # Resize both images to target size
    wm_img = tf.image.resize(wm_img, img_size)
    clean_img = tf.image.resize(clean_img, img_size)
    
    # Normalize from [0, 255] to [-1, 1]
    # This range works well with tanh activation in output layer
    wm_img = (tf.cast(wm_img, tf.float32) / 127.5) - 1.0
    clean_img = (tf.cast(clean_img, tf.float32) / 127.5) - 1.0
    
    return wm_img, clean_img

In [ ]:
# ============================================================================
# DATA AUGMENTATION
# ============================================================================
def augment_image_pair(wm_img, clean_img):
    """
    Apply identical random augmentations to both images in a pair.
    
    Augmentations:
    - Random horizontal flip (50% chance)
    - Random vertical flip (50% chance)
    - Random rotation (0°, 90°, 180°, 270°)
    
    Why these augmentations?
    - Geometric transformations preserve watermark patterns
    - Watermarks can appear in any orientation
    - Increases effective dataset size by ~8x
    - NO color/brightness changes (we want exact color reproduction)
    
    Args:
        wm_img: Watermarked image tensor
        clean_img: Clean image tensor
    
    Returns:
        (wm_img, clean_img): Augmented image pair
    """
    # Random horizontal flip
    if tf.random.uniform(()) > 0.5:
        wm_img = tf.image.flip_left_right(wm_img)
        clean_img = tf.image.flip_left_right(clean_img)
    
    # Random vertical flip
    if tf.random.uniform(()) > 0.5:
        wm_img = tf.image.flip_up_down(wm_img)
        clean_img = tf.image.flip_up_down(clean_img)
    
    # Random 90-degree rotation
    k = tf.random.uniform(shape=[], minval=0, maxval=4, dtype=tf.int32)
    wm_img = tf.image.rot90(wm_img, k=k)
    clean_img = tf.image.rot90(clean_img, k=k)
    
    return wm_img, clean_img

In [ ]:
# ============================================================================
# CREATE TENSORFLOW DATASETS (TPU-OPTIMIZED)
# ============================================================================

# Get full paths for training data
train_watermarked = [os.path.join(DATASET_DIR, p) for p in train_df['watermark_path'].values]
train_clean = [os.path.join(DATASET_DIR, p) for p in train_df['clean_path'].values]

# Get full paths for validation data
val_watermarked = [os.path.join(DATASET_DIR, p) for p in test_df['watermark_path'].values]
val_clean = [os.path.join(DATASET_DIR, p) for p in test_df['clean_path'].values]

# Training dataset
train_dataset = tf.data.Dataset.from_tensor_slices((train_watermarked, train_clean))
train_dataset = train_dataset.shuffle(len(train_df), reshuffle_each_iteration=True)
train_dataset = train_dataset.map(
    lambda wm, clean: augment_pair(*load_image_pair(wm, clean, IMG_SIZE)),
    num_parallel_calls=tf.data.AUTOTUNE
)
train_dataset = train_dataset.batch(BATCH_SIZE, drop_remainder=True)  # TPU requires drop_remainder
train_dataset = train_dataset.prefetch(tf.data.AUTOTUNE)  # TPU optimization

# Validation dataset
val_dataset = tf.data.Dataset.from_tensor_slices((val_watermarked, val_clean))
val_dataset = val_dataset.map(
    lambda wm, clean: load_image_pair(wm, clean, IMG_SIZE),
    num_parallel_calls=tf.data.AUTOTUNE
)
val_dataset = val_dataset.batch(BATCH_SIZE, drop_remainder=True)  # TPU requires drop_remainder
val_dataset = val_dataset.prefetch(tf.data.AUTOTUNE)  # TPU optimization

print(f"✓ Datasets created (TPU-optimized)")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Training batches: {len(train_df) // BATCH_SIZE}")
print(f"  Validation batches: {len(test_df) // BATCH_SIZE}")
print(f"  drop_remainder=True (required for TPU)")
print(f"  prefetch=AUTOTUNE (optimizes TPU pipeline)")

In [ ]:
# ============================================================================
# VERIFY DATASET PIPELINE
# ============================================================================
print("Testing dataset pipeline...")

# Get one batch from train dataset
for batch_wm, batch_clean in train_dataset.take(1):
    print(f"\nBatch shapes:")
    print(f"  Watermarked: {batch_wm.shape}  (batch, height, width, channels)")
    print(f"  Clean:       {batch_clean.shape}")
    print(f"\nValue ranges:")
    print(f"  Watermarked: [{batch_wm.numpy().min():.3f}, {batch_wm.numpy().max():.3f}]")
    print(f"  Clean:       [{batch_clean.numpy().min():.3f}, {batch_clean.numpy().max():.3f}]")
    print(f"\n✓ Dataset pipeline is working correctly!")
    
    # Visualize first image in batch
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    
    # Denormalize from [-1, 1] to [0, 1] for display
    wm_display = (batch_wm[0].numpy() + 1.0) / 2.0
    clean_display = (batch_clean[0].numpy() + 1.0) / 2.0
    
    axes[0].imshow(wm_display)
    axes[0].set_title('Watermarked (after preprocessing)')
    axes[0].axis('off')
    
    axes[1].imshow(clean_display)
    axes[1].set_title('Clean (after preprocessing)')
    axes[1].axis('off')
    
    plt.tight_layout()
    plt.show()

---
## 4. U-Net Model Architecture

### Why U-Net?

**Architecture Components:**
1. **Encoder (Downsampling)**: Captures features at multiple scales
   - 256×256 → 128×128 → 64×64 → 32×32 → 16×16
   - Each level detects progressively higher-level patterns

2. **Bottleneck**: Most compressed representation
   - 16×16 with 1024 channels
   - Understands global image structure and watermark location

3. **Decoder (Upsampling)**: Reconstructs clean image
   - 16×16 → 32×32 → 64×64 → 128×128 → 256×256
   - Receives skip connections from encoder

4. **Skip Connections**: The secret sauce!
   - Copy features from encoder to decoder
   - Preserve fine details (edges, textures)
   - Prevent blurry outputs

**Why it works for watermark removal:**
- Encoder learns to detect watermark patterns
- Bottleneck understands full image context
- Decoder reconstructs clean regions
- Skip connections preserve non-watermarked details

In [ ]:
# ============================================================================
# BUILDING BLOCKS: CONVOLUTIONAL BLOCK
# ============================================================================
def conv_block(x, filters, kernel_size=3, activation='relu', batch_norm=True):
    """
    Standard convolutional block: Conv → BatchNorm → Activation (×2)
    
    Structure:
        Conv2D(filters) → BatchNorm → ReLU
        Conv2D(filters) → BatchNorm → ReLU
    
    Args:
        x: Input tensor
        filters: Number of conv filters
        kernel_size: Convolution kernel size (default: 3×3)
        activation: Activation function (default: relu)
        batch_norm: Use batch normalization (default: True)
    
    Returns:
        Output tensor after two convolutions
    """
    # First convolution
    x = layers.Conv2D(
        filters, 
        kernel_size, 
        padding='same',              # Keep spatial dimensions
        kernel_initializer='he_normal'  # Good for ReLU
    )(x)
    
    if batch_norm:
        x = layers.BatchNormalization()(x)  # Normalize activations
    
    x = layers.Activation(activation)(x)
    
    # Second convolution
    x = layers.Conv2D(
        filters, 
        kernel_size, 
        padding='same',
        kernel_initializer='he_normal'
    )(x)
    
    if batch_norm:
        x = layers.BatchNormalization()(x)
    
    x = layers.Activation(activation)(x)
    
    return x

In [ ]:
# ============================================================================
# BUILDING BLOCKS: ENCODER BLOCK
# ============================================================================
def encoder_block(x, filters):
    """
    Encoder block: Convolutions → MaxPooling
    
    Flow:
        Input → Conv Block → [Skip Connection] → MaxPool → Output
                               ↓
                          (saved for decoder)
    
    Args:
        x: Input tensor
        filters: Number of filters for convolutions
    
    Returns:
        (conv, pool): 
            conv = Output before pooling (for skip connection)
            pool = Output after pooling (input to next layer)
    """
    # Apply convolutions
    conv = conv_block(x, filters)
    
    # Downsample with max pooling (2×2 = reduces size by half)
    pool = layers.MaxPooling2D(pool_size=(2, 2))(conv)
    
    return conv, pool

In [ ]:
# ============================================================================
# BUILDING BLOCKS: DECODER BLOCK
# ============================================================================
def decoder_block(x, skip_features, filters):
    """
    Decoder block: Upsample → Concatenate with skip → Convolutions
    
    Flow:
        Input → Upsample (2×) → Concat with Skip → Conv Block → Output
                                      ↑
                          (from encoder same level)
    
    Why concatenate?
    - Combines low-level details (skip) with high-level semantics (upsampled)
    - Preserves fine details lost during downsampling
    - Key to U-Net's success!
    
    Args:
        x: Input tensor from previous decoder layer
        skip_features: Features from corresponding encoder layer
        filters: Number of filters for convolutions
    
    Returns:
        Output tensor after upsampling and convolutions
    """
    # Upsample: 2× spatial dimensions using transposed convolution
    x = layers.Conv2DTranspose(
        filters, 
        (2, 2),              # Kernel size
        strides=(2, 2),      # Stride = 2 doubles spatial dimensions
        padding='same'
    )(x)
    
    # Concatenate with skip connection from encoder
    # Doubles the number of channels
    x = layers.Concatenate()([x, skip_features])
    
    # Apply convolutions
    x = conv_block(x, filters)
    
    return x

In [ ]:
# ============================================================================
# U-NET MODEL ARCHITECTURE
# ============================================================================
def build_unet(input_shape=(256, 256, 3), start_filters=64):
    """
    Build complete U-Net model for watermark removal.
    
    Architecture (with start_filters=64):
    
        Input (256×256×3)
           |
        Encoder:
           ├─ Block 1: 256×256×64  ──┐
           ├─ Block 2: 128×128×128 ──┤
           ├─ Block 3: 64×64×256   ──┤ Skip connections
           └─ Block 4: 32×32×512   ──┤
                                     │
        Bottleneck: 16×16×1024       │
                                     │
        Decoder:                     │
           ┌─ Block 4: 32×32×512   ←─┘
           ├─ Block 3: 64×64×256   ←──┘
           ├─ Block 2: 128×128×128 ←───┘
           └─ Block 1: 256×256×64  ←────┘
           |
        Output (256×256×3)
    
    Args:
        input_shape: Input image shape (height, width, channels)
        start_filters: Number of filters in first layer (doubled each level)
    
    Returns:
        Keras Model ready for training
    """
    # Input layer
    inputs = layers.Input(shape=input_shape, name='input_image')
    
    # ========================================================================
    # ENCODER (Downsampling path)
    # ========================================================================
    # Level 1: 256×256 → 128×128
    conv1, pool1 = encoder_block(inputs, start_filters)  # 64 filters
    
    # Level 2: 128×128 → 64×64
    conv2, pool2 = encoder_block(pool1, start_filters * 2)  # 128 filters
    
    # Level 3: 64×64 → 32×32
    conv3, pool3 = encoder_block(pool2, start_filters * 4)  # 256 filters
    
    # Level 4: 32×32 → 16×16
    conv4, pool4 = encoder_block(pool3, start_filters * 8)  # 512 filters
    
    # ========================================================================
    # BOTTLENECK (Most compressed representation)
    # ========================================================================
    # 16×16×1024
    bottleneck = conv_block(pool4, start_filters * 16)  # 1024 filters
    
    # ========================================================================
    # DECODER (Upsampling path)
    # ========================================================================
    # Level 4: 16×16 → 32×32 (concatenate with conv4)
    dec4 = decoder_block(bottleneck, conv4, start_filters * 8)  # 512 filters
    
    # Level 3: 32×32 → 64×64 (concatenate with conv3)
    dec3 = decoder_block(dec4, conv3, start_filters * 4)  # 256 filters
    
    # Level 2: 64×64 → 128×128 (concatenate with conv2)
    dec2 = decoder_block(dec3, conv2, start_filters * 2)  # 128 filters
    
    # Level 1: 128×128 → 256×256 (concatenate with conv1)
    dec1 = decoder_block(dec2, conv1, start_filters)  # 64 filters
    
    # ========================================================================
    # OUTPUT LAYER
    # ========================================================================
    # 1×1 convolution to get 3 channels (RGB)
    # tanh activation outputs values in [-1, 1]
    outputs = layers.Conv2D(
        3,                    # RGB channels
        (1, 1),              # 1×1 kernel
        activation='tanh',   # Output range: [-1, 1]
        padding='same',
        name='output_image'
    )(dec1)
    
    # Create model
    model = models.Model(inputs, outputs, name='UNet_WatermarkRemoval')
    
    return model

# ============================================================================
# BUILD MODEL
# ============================================================================
print("Building U-Net model...")

# Create model within TPU strategy scope
with strategy.scope():
    model = build_unet(
    input_shape=(*CONFIG['IMAGE_SIZE'], 3),
    start_filters=64
    )

    print("\n✓ Model built successfully!")
print(f"\nModel summary:")
model.summary()

---
## 5. Custom Loss Functions

### Why Multiple Loss Functions?

**Single L1 Loss (MAE):**
- ✓ Pixel-wise accuracy
- ✗ Often blurry outputs
- ✗ Doesn't capture perceptual quality

**Our Multi-Loss Approach:**

1. **L1 Loss (Weight: 1.0)**
   - Ensures pixel-level accuracy
   - Removes watermark precisely

2. **Perceptual Loss (Weight: 0.1)**
   - Uses VGG19 features
   - Ensures natural, realistic appearance
   - Prevents checkerboard artifacts

3. **SSIM Loss (Weight: 0.5)**
   - Structural Similarity Index
   - Preserves edges, textures, patterns
   - Better than MSE for perceptual quality

**Result:** Sharp, natural-looking images without artifacts!

In [ ]:
# ============================================================================
# PERCEPTUAL LOSS (VGG19-based) - WITH AUTOMATIC FALLBACK
# ============================================================================
print("Attempting to load VGG19 for perceptual loss...")
print("Note: VGG19 requires internet access to download weights (~550 MB)")
print("      If internet is disabled, model will use simplified loss.\n")

# Try to load VGG19 (requires internet access)
USE_PERCEPTUAL_LOSS = False

try:
    # Load pre-trained VGG19 (trained on ImageNet)
    vgg = tf.keras.applications.VGG19(
        include_top=False,              # Exclude classification layers
        weights='imagenet',             # Use ImageNet weights (downloads if needed)
        input_shape=(*CONFIG['IMAGE_SIZE'], 3)
    )
    vgg.trainable = False  # Freeze weights (we only use it for feature extraction)
    
    # Select layers for feature extraction
    # These layers capture different levels of visual features:
    #   - block1_conv2: Low-level (edges, colors)
    #   - block2_conv2: Mid-level (textures)
    #   - block3_conv3: Higher-level (patterns)
    #   - block4_conv3: High-level (objects, structures)
    perceptual_layers = ['block1_conv2', 'block2_conv2', 'block3_conv3', 'block4_conv3']
    
    # Create feature extraction model
    perceptual_model = models.Model(
        inputs=vgg.input,
        outputs=[vgg.get_layer(name).output for name in perceptual_layers],
        name='VGG19_Features'
    )
    
    USE_PERCEPTUAL_LOSS = True
    print(f"✓ VGG19 loaded successfully with {len(perceptual_layers)} feature extraction layers")
    print("  Model will use: L1 + Perceptual + SSIM loss")
    
    def perceptual_loss(y_true, y_pred):
        """
        Calculate perceptual loss using VGG19 features.
        
        Process:
        1. Convert images from [-1, 1] to VGG19 input format
        2. Extract features from both images
        3. Calculate L1 distance between features
        4. Average across all layers
        
        Why it works:
        - VGG19 trained on ImageNet understands natural images
        - Matching features = matching visual appearance
        - Better than pixel matching for perceptual quality
        
        Args:
            y_true: Ground truth clean images
            y_pred: Predicted clean images
        
        Returns:
            Perceptual loss value
        """
        # Convert from [-1, 1] to [0, 255] for VGG19
        y_true_prep = (y_true + 1.0) * 127.5
        y_pred_prep = (y_pred + 1.0) * 127.5
        
        # VGG19 preprocessing (subtract ImageNet mean)
        y_true_prep = tf.keras.applications.vgg19.preprocess_input(y_true_prep)
        y_pred_prep = tf.keras.applications.vgg19.preprocess_input(y_pred_prep)
        
        # Extract features
        true_features = perceptual_model(y_true_prep)
        pred_features = perceptual_model(y_pred_prep)
        
        # Calculate L1 distance for each layer
        loss = 0.0
        for true_feat, pred_feat in zip(true_features, pred_features):
            loss += tf.reduce_mean(tf.abs(true_feat - pred_feat))
        
        # Average across layers
        return loss / len(perceptual_layers)

except Exception as e:
    # VGG19 loading failed (likely no internet access)
    USE_PERCEPTUAL_LOSS = False
    print("⚠ Could not load VGG19 (likely no internet access)")
    print(f"  Error: {str(e)[:100]}...")
    print("\n" + "="*70)
    print("USING SIMPLIFIED MODEL (No VGG19)")
    print("="*70)
    print("To enable VGG19 (better quality):")
    print("  1. Click 'Settings' → Toggle 'Internet' ON")
    print("  2. Save and restart notebook")
    print("  3. VGG19 will download automatically (~550 MB, one-time)")
    print("\nCurrent model will use: L1 + SSIM loss (still excellent quality!)")
    print("  Expected PSNR: 28-33 dB (vs 30-35 with VGG19)")
    print("  Expected SSIM: 0.88-0.93 (vs 0.90-0.95 with VGG19)")
    print("="*70 + "\n")

In [ ]:
# ============================================================================
# SSIM LOSS (Structural Similarity)
# ============================================================================
def ssim_loss(y_true, y_pred):
    """
    Calculate SSIM (Structural Similarity Index) loss.
    
    SSIM measures similarity considering:
    - Luminance (brightness)
    - Contrast
    - Structure (edges, patterns)
    
    Range: 0 (different) to 1 (identical)
    We use 1 - SSIM as loss (lower is better)
    
    Args:
        y_true: Ground truth images
        y_pred: Predicted images
    
    Returns:
        SSIM loss (0 = identical, 1 = completely different)
    """
    # Calculate SSIM (max_val=2.0 because images are in [-1, 1])
    ssim_value = tf.image.ssim(y_true, y_pred, max_val=2.0)
    
    # Convert to loss (higher SSIM = lower loss)
    return 1.0 - tf.reduce_mean(ssim_value)

In [ ]:
# ============================================================================
# COMBINED LOSS FUNCTION
# ============================================================================
def combined_loss(y_true, y_pred):
    """
    Combined loss function: L1 + SSIM (+ Perceptual if available)
    
    Base Components (always used):
    1. L1 Loss (MAE): Pixel-wise accuracy
       Weight: 1.0 (primary loss)
    
    2. SSIM Loss: Structural preservation
       Weight: 0.5 (maintains edges and textures)
    
    Optional Component (if VGG19 loaded):
    3. Perceptual Loss: Natural appearance
       Weight: 0.1 (helps prevent artifacts)
    
    Why these weights?
    - L1 (1.0): Primary objective is accurate reconstruction
    - SSIM (0.5): Strong influence for sharp edges
    - Perceptual (0.1): Subtle influence for naturalness (if available)
    
    Args:
        y_true: Ground truth clean images
        y_pred: Predicted clean images
    
    Returns:
        Combined loss value
    """
    # Calculate base losses (always available)
    l1 = tf.reduce_mean(tf.abs(y_true - y_pred))  # L1 loss (MAE)
    ssim = ssim_loss(y_true, y_pred)              # Structural similarity
    
    # Start with base combination
    total_loss = 1.0 * l1 + 0.5 * ssim
    
    # Add perceptual loss only if VGG19 is loaded
    if USE_PERCEPTUAL_LOSS:
        perceptual = perceptual_loss(y_true, y_pred)  # VGG19-based
        total_loss += 0.1 * perceptual
    
    return total_loss

print("✓ Loss functions defined")
print("
Loss components:")
print("  L1 Loss (MAE):       Weight = 1.0")
print("  SSIM Loss:           Weight = 0.5")
if USE_PERCEPTUAL_LOSS:
    print("  Perceptual Loss:     Weight = 0.1 (VGG19-based)")
else:
    print("  Perceptual Loss:     Disabled (VGG19 not available)")

---
## 6. Metrics and Model Compilation

In [ ]:
# ============================================================================
# CUSTOM METRICS
# ============================================================================
def psnr_metric(y_true, y_pred):
    """
    Peak Signal-to-Noise Ratio (PSNR) metric.
    
    Measures image quality in decibels (dB):
    - 20-25 dB: Poor quality
    - 25-30 dB: Good quality
    - 30-35 dB: Excellent quality
    - >35 dB: Outstanding quality
    
    Higher is better!
    """
    return tf.image.psnr(y_true, y_pred, max_val=2.0)

def ssim_metric(y_true, y_pred):
    """
    Structural Similarity Index (SSIM) metric.
    
    Range: 0 to 1
    - 0.80-0.85: Good similarity
    - 0.85-0.90: Very good similarity
    - 0.90-0.95: Excellent similarity
    - >0.95: Near-perfect similarity
    
    Higher is better!
    """
    return tf.image.ssim(y_true, y_pred, max_val=2.0)

print("✓ Metrics defined")
print("\nMetrics to track:")
print("  - MAE (Mean Absolute Error): Lower is better")
print("  - PSNR (Peak Signal-to-Noise Ratio): Higher is better")
print("  - SSIM (Structural Similarity): Higher is better")

In [ ]:
# ============================================================================
# MODEL COMPILATION WITH TPU STRATEGY
# ============================================================================
with strategy.scope():
    model.compile(
        optimizer=Adam(learning_rate=LEARNING_RATE),
        loss=combined_loss,
        metrics=[psnr_metric, ssim_metric]
    )

print("✓ Model compiled successfully")
print(f"  Strategy: {strategy.__class__.__name__}")
print(f"  Devices: {strategy.num_replicas_in_sync}")

In [ ]:
# ============================================================================
# CALLBACKS
# ============================================================================
print("Setting up training callbacks...")

callbacks = [
    # Save best model based on validation loss
    ModelCheckpoint(
        filepath='best_watermark_remover.h5',
        monitor='val_loss',           # Watch validation loss
        save_best_only=True,          # Only save when val_loss improves
        mode='min',                   # Lower loss is better
        verbose=1,
        save_weights_only=False       # Save full model
    ),
    
    # Stop training if no improvement
    EarlyStopping(
        monitor='val_loss',           # Watch validation loss
        patience=15,                  # Wait 15 epochs before stopping
        restore_best_weights=True,    # Restore best weights when stopping
        verbose=1,
        mode='min'
    ),
    
    # Reduce learning rate when stuck
    ReduceLROnPlateau(
        monitor='val_loss',           # Watch validation loss
        factor=0.5,                   # Reduce LR by half
        patience=5,                   # Wait 5 epochs before reducing
        min_lr=1e-7,                  # Don't go below this LR
        verbose=1,
        mode='min'
    )
]

print("\n✓ Callbacks configured:")
print("  1. ModelCheckpoint: Save best model")
print("  2. EarlyStopping: Stop if no improvement (patience=15)")
print("  3. ReduceLROnPlateau: Reduce LR when stuck (patience=5)")

---
## 7. Training

**Training will take 2-4 hours on GPU**

**What to expect:**
- Epochs 1-20: Rapid loss decrease (learning watermark patterns)
- Epochs 20-50: Slower improvement (refining details)
- Epochs 50-80: Fine-tuning (perceptual quality)
- Epochs 80+: Minimal gains (early stopping may trigger)

**Target metrics after training:**
- Loss: <0.06
- PSNR: >30 dB
- SSIM: >0.90

In [ ]:
# ============================================================================
# TRAIN THE MODEL
# ============================================================================
print("Starting training...")
print(f"\nTraining configuration:")
print(f"  Epochs: {CONFIG['EPOCHS']}")
print(f"  Batch size: {CONFIG['BATCH_SIZE']}")
print(f"  Steps per epoch: {len(train_df) // CONFIG['BATCH_SIZE']}")
print(f"  Validation steps: {len(test_df) // CONFIG['BATCH_SIZE']}")
print(f"\nEstimated time: 2-4 hours on GPU")
print("\n" + "="*70)

# Train!
history = model.fit(
    train_dataset,                    # Training data
    validation_data=val_dataset,      # Validation data
    epochs=CONFIG['EPOCHS'],          # Number of epochs
    callbacks=callbacks,              # Callbacks (save, early stop, LR schedule)
    verbose=1                         # Show progress bar
)

print("\n" + "="*70)
print("✓ Training completed!")

---
## 8. Training Visualization

In [ ]:
# ============================================================================
# PLOT TRAINING HISTORY
# ============================================================================
def plot_training_history(history):
    """
    Plot training metrics over epochs.
    
    Creates 4 subplots:
    1. Loss (train vs validation)
    2. MAE (train vs validation)
    3. PSNR (train vs validation) - Higher is better
    4. SSIM (train vs validation) - Higher is better
    """
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # Loss
    axes[0, 0].plot(history.history['loss'], label='Train Loss', linewidth=2)
    axes[0, 0].plot(history.history['val_loss'], label='Val Loss', linewidth=2)
    axes[0, 0].set_title('Loss Over Epochs', fontsize=14, fontweight='bold')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # MAE
    axes[0, 1].plot(history.history['mae'], label='Train MAE', linewidth=2)
    axes[0, 1].plot(history.history['val_mae'], label='Val MAE', linewidth=2)
    axes[0, 1].set_title('MAE Over Epochs (Lower is Better)', fontsize=14, fontweight='bold')
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('MAE')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    
    # PSNR
    axes[1, 0].plot(history.history['psnr_metric'], label='Train PSNR', linewidth=2)
    axes[1, 0].plot(history.history['val_psnr_metric'], label='Val PSNR', linewidth=2)
    axes[1, 0].set_title('PSNR Over Epochs (Higher is Better)', fontsize=14, fontweight='bold')
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylabel('PSNR (dB)')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
    axes[1, 0].axhline(y=30, color='green', linestyle='--', alpha=0.5, label='Target: 30 dB')
    
    # SSIM
    axes[1, 1].plot(history.history['ssim_metric'], label='Train SSIM', linewidth=2)
    axes[1, 1].plot(history.history['val_ssim_metric'], label='Val SSIM', linewidth=2)
    axes[1, 1].set_title('SSIM Over Epochs (Higher is Better)', fontsize=14, fontweight='bold')
    axes[1, 1].set_xlabel('Epoch')
    axes[1, 1].set_ylabel('SSIM')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)
    axes[1, 1].axhline(y=0.90, color='green', linestyle='--', alpha=0.5, label='Target: 0.90')
    
    plt.tight_layout()
    plt.savefig('training_history.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    # Print final metrics
    print("\nFinal Training Metrics:")
    print(f"  Loss: {history.history['loss'][-1]:.4f}")
    print(f"  MAE: {history.history['mae'][-1]:.4f}")
    print(f"  PSNR: {history.history['psnr_metric'][-1]:.2f} dB")
    print(f"  SSIM: {history.history['ssim_metric'][-1]:.4f}")
    
    print("\nFinal Validation Metrics:")
    print(f"  Loss: {history.history['val_loss'][-1]:.4f}")
    print(f"  MAE: {history.history['val_mae'][-1]:.4f}")
    print(f"  PSNR: {history.history['val_psnr_metric'][-1]:.2f} dB")
    print(f"  SSIM: {history.history['val_ssim_metric'][-1]:.4f}")

# Plot training history
plot_training_history(history)

---
## 9. Evaluation and Results

In [ ]:
# ============================================================================
# LOAD BEST MODEL
# ============================================================================
print("Loading best model from training...")

best_model = keras.models.load_model(
    'best_watermark_remover.h5',
    custom_objects={
        'combined_loss': combined_loss,
        'psnr_metric': psnr_metric,
        'ssim_metric': ssim_metric
    }
)

print("✓ Best model loaded")

In [ ]:
# ============================================================================
# EVALUATE ON VALIDATION SET
# ============================================================================
print("Evaluating model on validation set...\n")

results = best_model.evaluate(val_dataset, verbose=1)

print("\n" + "="*70)
print("FINAL VALIDATION RESULTS")
print("="*70)
print(f"Loss:  {results[0]:.4f}")
print(f"MAE:   {results[1]:.4f}")
print(f"PSNR:  {results[2]:.2f} dB")
print(f"SSIM:  {results[3]:.4f}")
print("="*70)

# Interpret results
print("\nPerformance Assessment:")
if results[2] > 30 and results[3] > 0.90:
    print("✓ EXCELLENT: Model exceeds target performance!")
elif results[2] > 28 and results[3] > 0.88:
    print("✓ GOOD: Model performs well, consider training longer")
else:
    print("⚠ FAIR: Consider adjusting hyperparameters or training longer")

In [ ]:
# ============================================================================
# VISUALIZE PREDICTIONS
# ============================================================================
def show_predictions(model, dataset, num_samples=5):
    """
    Show watermark removal results side-by-side.
    
    Displays:
    - Input (watermarked)
    - Predicted (model output)
    - Ground truth (clean)
    
    Plus PSNR and SSIM scores for each prediction.
    """
    fig, axes = plt.subplots(num_samples, 3, figsize=(15, 5*num_samples))
    
    # Get batch from dataset
    for batch_wm, batch_clean in dataset.take(1):
        # Predict on watermarked images
        predictions = model.predict(batch_wm[:num_samples], verbose=0)
        
        for i in range(num_samples):
            # Denormalize images from [-1, 1] to [0, 1]
            wm_img = (batch_wm[i].numpy() + 1.0) / 2.0
            clean_img = (batch_clean[i].numpy() + 1.0) / 2.0
            pred_img = (predictions[i] + 1.0) / 2.0
            
            # Calculate metrics for this sample
            psnr = tf.image.psnr(
                batch_clean[i:i+1], 
                predictions[i:i+1], 
                max_val=2.0
            ).numpy()[0]
            
            ssim = tf.image.ssim(
                batch_clean[i:i+1], 
                predictions[i:i+1], 
                max_val=2.0
            ).numpy()[0]
            
            # Display input (watermarked)
            axes[i, 0].imshow(wm_img)
            axes[i, 0].set_title('Input\n(Watermarked)', fontsize=12, fontweight='bold')
            axes[i, 0].axis('off')
            
            # Display prediction
            axes[i, 1].imshow(pred_img)
            axes[i, 1].set_title(
                f'Predicted\nPSNR: {psnr:.2f}dB | SSIM: {ssim:.3f}', 
                fontsize=12, 
                fontweight='bold'
            )
            axes[i, 1].axis('off')
            
            # Display ground truth (clean)
            axes[i, 2].imshow(clean_img)
            axes[i, 2].set_title('Ground Truth\n(Clean)', fontsize=12, fontweight='bold')
            axes[i, 2].axis('off')
    
    plt.tight_layout()
    plt.savefig('predictions.png', dpi=300, bbox_inches='tight')
    plt.show()

print("Generating predictions on validation set...\n")
show_predictions(best_model, val_dataset, num_samples=5)

---
## 10. Export Model for Production

In [ ]:
# ============================================================================
# EXPORT MODELS IN MULTIPLE FORMATS
# ============================================================================
print("Exporting trained model in multiple formats...\n")

# 1. Keras H5 format (already saved)
print("✓ Keras H5 format: best_watermark_remover.h5")

# 2. TensorFlow SavedModel format (for TensorFlow Serving)
print("\nExporting TensorFlow SavedModel...")
best_model.save('watermark_remover_savedmodel', save_format='tf')
print("✓ TensorFlow SavedModel: watermark_remover_savedmodel/")

# 3. TensorFlow Lite format (for mobile/edge deployment)
print("\nExporting TensorFlow Lite model...")
converter = tf.lite.TFLiteConverter.from_keras_model(best_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]  # Optimize for size
tflite_model = converter.convert()

with open('watermark_remover.tflite', 'wb') as f:
    f.write(tflite_model)
print("✓ TensorFlow Lite: watermark_remover.tflite")

# Print model sizes
import os
h5_size = os.path.getsize('best_watermark_remover.h5') / (1024 * 1024)
tflite_size = os.path.getsize('watermark_remover.tflite') / (1024 * 1024)

print("\n" + "="*70)
print("MODEL EXPORT SUMMARY")
print("="*70)
print(f"Keras H5:              {h5_size:.1f} MB  (For Python/FastAPI backend)")
print(f"TensorFlow SavedModel: {h5_size:.1f} MB  (For TF Serving)")
print(f"TensorFlow Lite:       {tflite_size:.1f} MB  (For mobile/edge devices)")
print("="*70)

print("\n✓ All models exported successfully!")

---
## 11. Inference Function for Production

In [ ]:
# ============================================================================
# PRODUCTION INFERENCE FUNCTION
# ============================================================================
def remove_watermark(model, image_path, output_path='output_clean.jpg'):
    """
    Remove watermark from a single image.
    
    Process:
    1. Load image (any size)
    2. Resize to model input size (256×256)
    3. Normalize to [-1, 1]
    4. Run model inference
    5. Denormalize to [0, 255]
    6. Resize back to original size
    7. Save cleaned image
    
    Args:
        model: Trained watermark removal model
        image_path: Path to watermarked image
        output_path: Path to save cleaned image
    
    Returns:
        PIL Image of cleaned result
    """
    # Load image
    img = Image.open(image_path).convert('RGB')
    original_size = img.size  # Save original size
    
    # Resize to model input size
    img_resized = img.resize(CONFIG['IMAGE_SIZE'])
    img_array = np.array(img_resized, dtype=np.float32)
    
    # Normalize to [-1, 1]
    img_normalized = (img_array / 127.5) - 1.0
    
    # Add batch dimension
    img_batch = np.expand_dims(img_normalized, axis=0)
    
    # Run inference
    pred = model.predict(img_batch, verbose=0)[0]
    
    # Denormalize to [0, 255]
    pred_img = ((pred + 1.0) * 127.5).astype(np.uint8)
    
    # Resize back to original size
    pred_pil = Image.fromarray(pred_img)
    pred_pil_resized = pred_pil.resize(original_size, Image.LANCZOS)
    
    # Save cleaned image
    pred_pil_resized.save(output_path, quality=95)
    print(f"✓ Cleaned image saved to: {output_path}")
    
    return pred_pil_resized

# ============================================================================
# TEST INFERENCE ON SAMPLE IMAGE
# ============================================================================
print("Testing inference function...\n")

# Get path to first test image
test_image_path = os.path.join(DATASET_DIR, test_df.iloc[0]['watermark_path'])

# Remove watermark
result = remove_watermark(best_model, test_image_path, 'test_output.jpg')

# Display results
fig, axes = plt.subplots(1, 2, figsize=(14, 7))

axes[0].imshow(Image.open(test_image_path))
axes[0].set_title('Original (Watermarked)', fontsize=14, fontweight='bold')
axes[0].axis('off')

axes[1].imshow(result)
axes[1].set_title('Cleaned (Model Output)', fontsize=14, fontweight='bold')
axes[1].axis('off')

plt.tight_layout()
plt.show()

print("\n✓ Inference function tested successfully!")

---
## 12. Final Summary

### ✅ Training Complete!

**What We Built:**
- U-Net architecture with skip connections
- Multi-loss function (L1 + Perceptual + SSIM)
- Data augmentation pipeline
- Comprehensive evaluation metrics

**Model Outputs:**
- `best_watermark_remover.h5` (Keras format)
- `watermark_remover_savedmodel/` (TensorFlow Serving)
- `watermark_remover.tflite` (Mobile deployment)
- `training_history.png` (Training curves)
- `predictions.png` (Sample results)

**Next Steps:**
1. Download trained model from Kaggle Output
2. Integrate into your FastAPI backend
3. Test on your own watermarked images
4. Deploy to production!

**Performance Tips:**
- If results are blurry: Increase perceptual loss weight
- If edges are soft: Increase SSIM loss weight
- If artifacts appear: Train longer or add more augmentation
- For better quality: Try larger image size (512×512)

**Citation:**
```
U-Net: Convolutional Networks for Biomedical Image Segmentation
Ronneberger et al., 2015
```

---

**Thank you for using this notebook! 🎉**

For questions or improvements, refer to:
- `README.md` - Overview and architecture
- `WHY_IT_WORKS.md` - Technical deep-dive
- `TROUBLESHOOTING.md` - Common issues and solutions